# Apriori sample-data validation

Build a small reproducible basket sample, explain association metrics, and compare sample results with the full store-period dataset.


## Imports


In [ ]:
import os
import pyodbc
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy.engine import URL
from mlxtend.frequent_patterns import apriori, association_rules
from sqlalchemy import create_engine, text


## Configure paths and the SQL Server source


In [ ]:
# Load environment variables from the project root when available.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

datasets_dir = project_root / "datasets"
env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv()

dw_user = (os.getenv("DATAWAREHOUSE_USER") or "").strip()
dw_password = (os.getenv("DATAWAREHOUSE_PASSWORD") or "").strip()
dw_host = (os.getenv("DATAWAREHOUSE_HOST") or "").strip()
dw_database = (os.getenv("DATAWAREHOUSE_DATABASE") or "").strip()

if not all([dw_user, dw_password, dw_host, dw_database]):
    raise ValueError("Missing one or more DATAWAREHOUSE_* environment variables.")

drivers = pyodbc.drivers()
print("Available ODBC drivers:", drivers)

preferred_driver_names = [
    (os.getenv("SQLSERVER_DRIVER") or "").strip(),
    "ODBC Driver 18 for SQL Server",
    "ODBC Driver 17 for SQL Server",
    "SQL Server",
]

driver = next(
    (
        candidate
        for candidate in preferred_driver_names
        if candidate and any(candidate.lower() in d.lower() for d in drivers)
    ),
    None,
)

if not driver:
    raise RuntimeError(
        f"No supported SQL Server ODBC driver found. Available drivers: {drivers}. "
        "Install Microsoft ODBC Driver 18 for SQL Server and verify the 64-bit ODBC Administrator."
    )

print(f"Using SQL Server driver: {driver}")

connection_string = URL.create(
    drivername="mssql+pyodbc",
    username=dw_user,
    password=dw_password,
    host=dw_host,
    database=dw_database,
    query={
        "driver": driver,
        "Encrypt": "yes",
        "TrustServerCertificate": "yes",
        "LoginTimeout": "30",
    },
)

engine = create_engine(connection_string, pool_pre_ping=True, future=True)

with engine.connect() as conn:
    print("Database connection OK:", conn.execute(text("SELECT 1")).scalar())

query = text(
    """
    SELECT 
        dd.[date] AS TRX_date,
        [StoreCode],
        [BillNo],
        [ItemCode],
        ITEMLONGNAME,
        [Quantity],
        DEPARTMENT,
        CLASS,
        SUBCLASS,
        [UOM_CD],
        [TotalAmt],
        [NetValue],
        [WACValue],
        [CSM_QTY],
        [CONSIGN_FINAL_QTY],
        [POS_FINAL_QTY]
    FROM [DBWH_8555].[dbo].[FactSalesTrxNew] fstn
    INNER JOIN dimdate dd ON dd.datekey = fstn.DateKey
    INNER JOIN DimItem di ON fstn.ItemCode = di.ITMCD
    WHERE dd.[date] BETWEEN :start_date AND :end_date AND StoreCode = '002'
    """)


## Load and validate source transactions


In [ ]:
with engine.connect() as conn:
    start_date = '2025-07-01'
    end_date = '2025-07-31'

    df = pd.read_sql(query, conn,  params={'start_date': start_date, 'end_date': end_date})
    
df = df[~df['ITEMLONGNAME'].str.contains('pepito bag', case=False, na=False)]
#df['month_year'] = pd.to_datetime(df['month_year'], format='%m-%Y')
#df['rule'] = df['antecedents'] + " → " + df['consequents']

df.info()

In [ ]:
df.head(1)

In [ ]:
unique_bill_no = df.BillNo.nunique()
unique_item_no = df.ITEMLONGNAME.nunique()


print(unique_bill_no)
print(unique_item_no)



## Select products for the sample


In [ ]:
# Get top 10 most frequent products for this store
product_frequency = df.groupby('ItemCode').size().sort_values(ascending=False)
top_products = product_frequency.head(10).index
bottom_products = product_frequency.tail(10).index

store_df_filtered = df[df['ItemCode'].isin(top_products)]

group_item = store_df_filtered.groupby('ItemCode').size().sort_values(ascending=False)

group_item

## Build binary transaction baskets


In [ ]:
transactions = store_df_filtered.groupby(['BillNo', 'ItemCode'])['POS_FINAL_QTY'] \
                                .sum().unstack(fill_value=0)

transactions = transactions.apply(pd.to_numeric, errors='coerce').fillna(0)
transactions_binary = transactions.gt(0)  # Convert to boolean (True/False)

transactions_binary.info()

In [ ]:
transactions_binary.head(100)

t10 = transactions_binary.head(10)

#convert to csv
transactions_binary.to_csv(datasets_dir / "transactions_binary.csv", index=False)

t10


## Run Apriori on the small sample


In [ ]:
# Generate frequent itemsets
# pd.set_option('display.max_colwidth', None)
frequent_itemsets_test = apriori(
    t10,
    min_support=0.001, # 0.001 = Only include itemsets appearing in ≥ 0.1% of transactions
    use_colnames=True,
)

frequent_itemsets_test

## Interpret support, confidence, and lift

### Support = (number of transactions containing the item) / (total number of transactions)

Total transaction of AQUA MINERAL WATER 1500ML is 1 = 1/10 = 0.1

Only get 2 itemsets, with 2 items each, possible apriori rules = 4

##### Confidence(A→B)= Support(A∪B)/Support(A)
Confidence asks:
“How often does B appear when A appears?”



-r1 = support(AQUA MINERAL WATER 600ML, AQUA MINERAL WATER 1500ML) / support(AQUA MINERAL WATER 600ML)
    = 0.1/0.1
    = 1

-r2 = support(AQUA MINERAL WATER 1500ML, AQUA MINERAL WATER 600ML) / support(AQUA MINERAL WATER 1500ML)
    = 0.1/0.1
    = 1

-r3 = support(FRUIT CUT 30K, SUNPRIDE PISANG CAVENDISH) / support(FRUIT CUT 30K)
    = 0.1/0.2
    = 0.5

-r4 = support(SUNPRIDE PISANG CAVENDISH, FRUIT CUT 30K) / support(SUNPRIDE PISANG CAVENDISH)
    = 0.1/0.5
    = 0.2


##### Lift(A→B)= Confidence(A→B)/Support(B)
Lift asks:
“How much more likely is B when A happens, compared to random chance of B?”

-r1 = confidence(AQUA MINERAL WATER 600ML, AQUA MINERAL WATER 1500ML) / support(AQUA MINERAL WATER 1500ML)
    = 1/0.1
    = 10

-r2 = confidence(AQUA MINERAL WATER 1500ML, AQUA MINERAL WATER 600ML) / support(AQUA MINERAL WATER 600ML)
    = 1/0.1
    = 10

-r3 = confidence(FRUIT CUT 30K, SUNPRIDE PISANG CAVENDISH) / support(SUNPRIDE PISANG CAVENDISH)
    = 0.5/0.5
    = 1

-r4 = confidence(SUNPRIDE PISANG CAVENDISH, FRUIT CUT 30K) / support(FRUIT CUT 30K)
    = 0.2/0.2
    = 1


In [ ]:
rules_test = association_rules(frequent_itemsets_test, metric="confidence", min_threshold=0.1)
rules_test

## Run Apriori on the full basket matrix


In [ ]:
#total trx: 6490 
pd.set_option('display.max_colwidth', None)

frequent_itemsets = apriori(
    transactions_binary,
    min_support=0.001, # 0.001 = Only include itemsets appearing in ≥ 0.1% of transactions
    use_colnames=True,
)


rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)

code_to_name = dict(zip(df['ItemCode'], df['ITEMLONGNAME']))

def codes_to_names(codes):
    return ', '.join(code_to_name.get(code, code) for code in codes)

rules['antecedents_names'] = rules['antecedents'].apply(codes_to_names)
rules['consequents_names'] = rules['consequents'].apply(codes_to_names)

rules = rules [['antecedents', 'antecedents_names','consequents',
       'consequents_names', 'antecedent support',
       'consequent support', 'support', 'confidence', 'lift',
       'representativity', 'leverage', 'conviction', 'zhangs_metric',
       'jaccard', 'certainty', 'kulczynski']]


rules['antecedents'] = rules['antecedents'].apply (lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply (lambda x: ', '.join(list(x)))

rules.to_csv(datasets_dir / "product_associations.csv", index=False)



#frequent_itemsets.itemsets = frequent_itemsets.itemsets.apply (lambda x: ', '.join(list(x)))



rules_sort = rules.sort_values(by='confidence', ascending=False) 
#rules_filter = rules_sort[rules_sort['consequents'].str.contains('RTE SALAD BUFFET PEPITO', case=False, na=False)]

rules_sort
#frequent_itemsets


In [ ]:
print(rules.columns)

'''

antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 
'representativity', 'leverage', 'conviction', 'zhangs_metric','jaccard', 'certainty', 'kulczynski', 'store_code', 'month_year'

'''

## Validate named-item support


In [ ]:
pair = ['CHICKEN BREAST BONELESS KG', 'BROKOLI LOKAL KG', 'SUNPRIDE PISANG CAVENDISH', 'DRAGON FRUIT MERAH', 'WORTEL MEDAN KG']

# convert item code to item name
transactions_with_names = transactions_binary.rename(columns=code_to_name)

transactions_with_names.to_csv(datasets_dir / "transactions_with_names.csv")


# Check dimensions
print(f"Transactions total: {len(transactions_with_names)}")
print(f"Columns in binary: {transactions_with_names.columns.nunique()} items")

# Check how many transactions contain both items
if all(col in transactions_with_names.columns for col in pair):
    count = transactions_with_names[pair].all(axis=1).sum()
    support = count / len(transactions_with_names)
    print(f"Support from binary: {count} / {len(transactions_with_names)} = {support:.6f}")
else:
    print("❌ One or both item names not found in transactions_binary.")


In [ ]:
# Re-run Apriori
frequent_itemsets = apriori(
    transactions_with_names,
    min_support=0.001,
    use_colnames=True
)

# Filter for the pair
mask = frequent_itemsets['itemsets'].apply(lambda x: x == frozenset(pair))
print(frequent_itemsets[mask])